# 🇹🇷 Kendi Türkçe Çeviri Motorunu Eğit

Bu defter Google Colab'da tamamen ücretsiz çalışır.  
Büyük OPUS paralel korpuslarını indirip **kendi SMT motorunu** eğitir.

## Özellikler
- ✅ **Witten-Bell trigram dil modeli** (Laplace'tan 3-4 BLEU puan daha iyi)
- ✅ **Ters öbek olasılığı** φ(ē|f̄) — ek ayırt edici güç
- ✅ **7-kelimeye kadar öbek** (Türkçe bileşik yapılar için)
- ✅ **Veri kalite filtresi** — gürültülü çiftleri otomatik eler
- ✅ **Çok çekirdekli eğitim** — Colab CPU'larını tam kullan
- ✅ **BLEU değerlendirme + ağırlık ayarı** (MERT)

## Hızlı başlangıç
1. **Çalışma Zamanı → Tüm hücreleri çalıştır** (ya da Ctrl+F9)
2. İstediğin korpusu seç (aşağıda `CORPUS_CHOICE`)
3. ~30-90 dakika bekle
4. `model.json.gz` indir → `egit.html`'e yükle veya `cevir-kendi.html`'de kullan

---

## 1. Ayarlar — Buradan Başla

In [ ]:
# ============================================================
#  AYARLAR — İHTİYACINA GÖRE DEĞİŞTİR
# ============================================================

# Hangi korpus(lar)ı kullanmak istiyorsun?
# Küçükten büyüğe: tatoeba (~10k), ted (~210k), qed (~380k),
#                  wikimatrix (~1.5M), opensubtitles (~5M), ccmatrix (~25M)
# Birden fazla için aralarına boşluk koy.
CORPORA = "ted tatoeba"        # Hızlı test (~15 dk)
# CORPORA = "ted tatoeba wikimatrix"  # İyi kalite (~45 dk, ~1.7M çift)
# CORPORA = "ted tatoeba wikimatrix opensubtitles"  # Çok iyi (~2 saat, ~6M çift)

# Her korpustan maksimum kaç çift al? 0 = sınırsız
LIMIT_PER_CORPUS = 0

# Eğitim parametreleri
ITER       = 15     # IBM-1 EM iterasyon sayısı (10-20 arasında iyi)
MINCOUNT   = 2      # Öbek min. gözlem sayısı (büyük veri için 3-5 öner)
MAXPHRASE  = 7      # Maksimum öbek uzunluğu (7 standart, büyük veri için ideal)
STEM       = True   # Türkçe köke indirgeme (hizalama için; çıktıyı bozmaz)

# Ağırlık ayarı (MERT) — dev seti varsa True yap
DO_TUNE    = True
DEV_TSV    = "data/dev-ornek.tsv"  # kaynak<TAB>hedef çiftleri

# Çıktı
OUT_MODEL  = "model.json.gz"

print(f"Ayarlar:")
print(f"  Korpus(lar): {CORPORA}")
print(f"  İterasyon: {ITER}, MinCount: {MINCOUNT}, MaxPhrase: {MAXPHRASE}")
print(f"  Stem: {STEM}, Tune: {DO_TUNE}")
print(f"  Çıktı: {OUT_MODEL}")

## 2. Node.js + Repo Kurulumu

In [ ]:
%%bash
# Node.js 20 LTS kur
if ! node --version 2>/dev/null | grep -q '^v2[0-9]'; then
  echo '=== Node.js 20 kuruluyor...'
  curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null 2>&1
  apt-get install -y nodejs > /dev/null 2>&1
fi
echo "Node: $(node --version)  npm: $(npm --version)"

In [ ]:
%%bash
# Repo'yu YENİ branch'ten klon veya güncelle
REPO="https://github.com/bilgekaan-tr/-eviri.git"
BRANCH="claude/translation-engine-product-lzz5co"
DIR="/content/-eviri"
if [ -d "$DIR/.git" ]; then
  echo '=== Mevcut repo güncelleniyor...'
  cd "$DIR" && git fetch --quiet origin && git checkout "$BRANCH" --quiet && git pull --quiet origin "$BRANCH"
else
  echo '=== Repo klonlanıyor (branch: '$BRANCH')...'
  git clone --quiet --branch "$BRANCH" "$REPO" "$DIR"
fi
echo "Dal: $(cd $DIR && git branch --show-current)"
echo "Son commit: $(cd $DIR && git log -1 --oneline)"

In [ ]:
%%bash
# Bağımlılıkları yükle (sadece eğitim için zaten dış bağımlılık yok,
# ama package.json var)
cd /content/-eviri && npm install --silent 2>&1 | tail -3
echo "npm install: OK"

## 3. Paralel Korpus İndir

In [ ]:
import subprocess, os, shlex

os.chdir("/content/-eviri")
corpora = CORPORA.split()
tsv_files = []

for corp in corpora:
    tsv_out = f"/content/corpus_{corp}.tsv"
    if os.path.exists(tsv_out):
        print(f"⏭  {corp}: zaten indirilmiş ({tsv_out})")
        tsv_files.append(tsv_out)
        continue
    cmd = ["node", "scripts/mt-fetch-corpus.js",
           "--corpus", corp, "--out", tsv_out]
    if LIMIT_PER_CORPUS > 0:
        cmd += ["--limit", str(LIMIT_PER_CORPUS)]
    print(f"\n⬇️  {corp} indiriliyor...")
    result = subprocess.run(cmd, capture_output=False, text=True)
    if result.returncode == 0:
        tsv_files.append(tsv_out)
    else:
        print(f"  HATA: {corp} indirilemedi")

print(f"\n✅ İndirilen: {tsv_files}")

In [ ]:
# TSV dosyalarını birleştir
import os

merged_tsv = "/content/korpus_birlesik.tsv"
total_lines = 0

with open(merged_tsv, "w", encoding="utf-8") as out_f:
    for tsv in tsv_files:
        with open(tsv, "r", encoding="utf-8") as in_f:
            for line in in_f:
                out_f.write(line)
                total_lines += 1
        print(f"  {os.path.basename(tsv)}: eklendi")

size_mb = os.path.getsize(merged_tsv) / 1048576
print(f"\n📦 Birleşik korpus: {total_lines:,} çift  ({size_mb:.1f} MB)  → {merged_tsv}")

## 4. Model Eğit

In [ ]:
import subprocess, multiprocessing, os, time

n_cpu = multiprocessing.cpu_count()
model_out = f"/content/{OUT_MODEL.replace('.gz','')}"

cmd = [
    "node", "scripts/mt-train-parallel.js",
    "--tsv", merged_tsv,
    "--out", model_out,
    "--workers", str(n_cpu),
    "--iter", str(ITER),
    "--mincount", str(MINCOUNT),
    "--maxphrase", str(MAXPHRASE),
    "--gzip",
]
if STEM:
    cmd.append("--stem")

print(f"🏋️  Eğitim başlıyor: {total_lines:,} çift, {n_cpu} çekirdek")
print(f"   Komut: {' '.join(cmd)}")
print()

t0 = time.time()
result = subprocess.run(cmd, text=True)
elapsed = time.time() - t0

model_gz = model_out + ".gz" if os.path.exists(model_out + ".gz") else model_out
if result.returncode == 0 and os.path.exists(model_gz):
    size_mb = os.path.getsize(model_gz) / 1048576
    print(f"\n✅ Model hazır: {model_gz}  ({size_mb:.1f} MB)  [{elapsed/60:.1f} dk]")
else:
    print(f"\n❌ Eğitim hatası (çıkış kodu: {result.returncode})")

## 5. BLEU Değerlendirme

In [ ]:
import subprocess, os

model_path = model_gz if 'model_gz' in dir() else f"/content/{OUT_MODEL}"
dev_path = f"/content/-eviri/{DEV_TSV}"

if not os.path.exists(dev_path):
    print(f"⚠️  Dev seti bulunamadı: {dev_path}")
    print("   Kendi dev setini şu formatta oluştur: kaynak<TAB>türkçe_referans")
else:
    cmd = ["node", "scripts/mt-eval.js",
           "--model", model_path,
           "--dev", dev_path,
           "--examples", "5"]
    print("📊 BLEU değerlendiriliyor...")
    result = subprocess.run(cmd, text=True, capture_output=False)
    if result.returncode != 0:
        print("❌ Değerlendirme hatası")

## 6. Ağırlık Ayarı (MERT)

In [ ]:
import subprocess, os

if not DO_TUNE:
    print("⏭  Ayarlama atlandı (DO_TUNE=False)")
elif not os.path.exists(dev_path):
    print(f"⚠️  Dev seti yok, ayarlama atlandı")
else:
    tuned_out = model_path.replace(".gz", "-ayarli.json.gz").replace(".json", "-ayarli.json")
    cmd = ["node", "scripts/mt-tune.js",
           "--model", model_path,
           "--dev", dev_path,
           "--out", tuned_out]
    print("⚙️  Ağırlık ayarı (MERT koordinat-yükseliş)...")
    result = subprocess.run(cmd, text=True, capture_output=False)
    if result.returncode == 0 and os.path.exists(tuned_out):
        print(f"\n✅ Ayarlı model: {tuned_out}")
        model_path = tuned_out  # bundan sonra ayarlı modeli kullan
    else:
        print("❌ Ayarlama hatası")

## 7. Test Çevirisi

In [ ]:
# Kendi cümlelerini dene
TEST_SENTENCES = [
    "Artificial intelligence is changing the world rapidly.",
    "The book was published in 1984 and became very popular.",
    "She went to the store to buy some bread and milk.",
    "This translation engine was built from scratch using statistical methods.",
    "The new algorithm achieves state-of-the-art performance on all benchmarks.",
]

import subprocess, os, tempfile

mp = model_path if 'model_path' in dir() else f"/content/{OUT_MODEL}"

print("🔤 Test çevirileri:\n")
for sent in TEST_SENTENCES:
    with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False, encoding='utf-8') as f:
        f.write(sent)
        tmp = f.name
    result = subprocess.run(
        ["node", "scripts/mt-translate.js", "--model", mp, "--text", sent],
        capture_output=True, text=True
    )
    os.unlink(tmp)
    tr = result.stdout.strip() or "(çeviri yok)"
    print(f"  EN: {sent}")
    print(f"  TR: {tr}")
    print()

## 8. Modeli İndir

In [ ]:
from google.colab import files
import os

mp = model_path if 'model_path' in dir() else f"/content/{OUT_MODEL}"

if os.path.exists(mp):
    size_mb = os.path.getsize(mp) / 1048576
    print(f"📥 İndiriliyor: {mp}  ({size_mb:.1f} MB)")
    print("   Kullanım: egit.html'de 'Model Yükle' veya cevir-kendi.html'de 'Model Yükle'")
    files.download(mp)
else:
    print(f"❌ Model bulunamadı: {mp}")
    print("   Önce eğitim hücrelerini çalıştır.")

---
## 📖 Notlar

### Kalite beklentisi
| Korpus | Çift | Eğitim süresi | Beklenen BLEU |
|--------|------|--------------|---------------|
| Tatoeba + TED | ~210k | ~15 dk | %8-12 |
| + WikiMatrix | ~1.7M | ~45 dk | %12-18 |
| + OpenSubtitles | ~6M | ~2 saat | %18-24 |

*(BLEU %20-25 = iyi bir istatistiksel MT; Google Translate ~%35-40 düzeyindedir.)*  
*Belirli bir alana (hukuk, tıp, edebiyat) ait ek eğitim verisi kaliteyi büyük ölçüde artırır.*

### Model nasıl kullanılır?
1. **Tarayıcı (cevir-kendi.html):** "Model Yükle" ile indirdiğin `.json.gz` dosyasını seç → PDF yükle → Çevir
2. **Eğitim arayüzü (egit.html):** Model yükle → Ek kitaplarla ince ayar yap
3. **CLI:** `node scripts/mt-translate.js --model model.json.gz --text "Hello world"`

### Birden fazla modeli birleştir
```bash
node scripts/mt-merge.js --models m1.json.gz m2.json.gz --out merged.json --gzip
```

### Kendi paralel metinlerinle eğit (kitap çifti)
```bash
node scripts/mt-train-parallel.js --src kitap_en.txt --tgt kitap_tr.txt --out kitap-model.json --gzip
```